### Import Dependencies

In [ ]:
import uuid

import httpx
from a2a.client import A2ACardResolver, create_client
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest
)
from google.protobuf.json_format import MessageToJson

### Test the connection

In [ ]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent-card.json"

In [ ]:
async with httpx.AsyncClient() as httpx_client:

    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL
    )

    try:
        print(f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}")
        public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(MessageToJson(public_card, indent=2))

    except Exception as e:
        print(f"Error fetching public agent card: {e}")

### Run commands against the Agent

In [ ]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await create_client(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.ROLE_USER,
    message_id=message_id,
    parts=[Part(text="Hello, how are you?")],
)
request = SendMessageRequest(message=message_payload)

async for response in client.send_message(request):
    print(response)

In [ ]:
async def run_a2a_warehouse_agent(query: str):

    timeout_config = httpx.Timeout(
        connect=10.0,   # Connection timeout
        read=100.0,     # Read timeout (important for long-running operations)
        write=10.0,     # Write timeout
        pool=10.0       # Pool timeout
    )

    client = await create_client(
        agent=BASE_URL,
        resolver_http_kwargs={"timeout": timeout_config}
    )


    message_id = str(uuid.uuid4())
    message_payload = Message(
        role=Role.ROLE_USER,
        message_id=message_id,
        parts=[Part(text=query)],
    )
    request = SendMessageRequest(message=message_payload)

    final_text_parts = []

    async for response in client.send_message(request):
        if response.HasField("artifact_update"):
            for part in response.artifact_update.artifact.parts:
                if part.text:
                    final_text_parts.append(part.text)

    final_response = "".join(final_text_parts)

    return final_response


In [ ]:
answer_1 = await run_a2a_warehouse_agent("What is the availability of B09X1LDMH6 in all of your warehouses?")

In [ ]:
print(answer_1)

In [ ]:
answer_1 = await run_a2a_warehouse_agent("can you reserve 1 of B09X1LDMH6 in Hamburg?")

In [ ]:
print(answer_1)